# word2vec from scratch

skipgram with negative sampling. text8 corpus.

rolled my own to actually understand it.


In [1]:
import torch
import torch.nn as nn
import numpy as np
from collections import Counter

# tiny corpus to start; will move to text8 once the loop works
text = '''the quick brown fox jumps over the lazy dog
the dog barks at the fox'''.lower().split()
print('tokens:', len(text))


In [2]:
class SGNS(nn.Module):
    def __init__(self, vocab, dim=64):
        super().__init__()
        self.in_emb = nn.Embedding(vocab, dim)
        self.out_emb = nn.Embedding(vocab, dim)
    def forward(self, center, ctx, neg):
        c = self.in_emb(center)
        o = self.out_emb(ctx)
        n = self.out_emb(neg)
        pos = torch.sigmoid((c * o).sum(-1))
        nega = torch.sigmoid(-(c.unsqueeze(1) * n).sum(-1))
        return -(torch.log(pos + 1e-9) + torch.log(nega + 1e-9).sum(-1)).mean()


In [3]:
vocab = sorted(set(text))
w2i = {w:i for i,w in enumerate(vocab)}
i2w = {i:w for w,i in w2i.items()}
print('vocab size:', len(vocab))


In [4]:
# build training pairs (center, ctx) for window=2
import numpy as np
win = 2
pairs = []
for i, w in enumerate(text):
    for j in range(max(0,i-win), min(len(text), i+win+1)):
        if i == j: continue
        pairs.append((w2i[w], w2i[text[j]]))
print('pairs:', len(pairs))


In [5]:
# negative sampling pool
freqs = Counter(text)
weights = np.array([freqs[w]**0.75 for w in vocab], dtype=np.float32)
weights /= weights.sum()


In [6]:
model = SGNS(len(vocab), dim=32)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for ep in range(200):
    losses = []
    np.random.shuffle(pairs)
    for c, o in pairs:
        neg_idx = np.random.choice(len(vocab), 5, p=weights)
        c_t = torch.tensor([c])
        o_t = torch.tensor([o])
        n_t = torch.tensor(neg_idx).unsqueeze(0)
        opt.zero_grad()
        loss = model(c_t, o_t, n_t)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    if ep % 25 == 0:
        print(ep, sum(losses)/len(losses))


In [7]:
# nearest neighbors after training
import torch.nn.functional as F
emb = model.in_emb.weight.detach()
emb = emb / emb.norm(dim=1, keepdim=True)
for q in ['fox', 'dog']:
    if q in w2i:
        sims = emb @ emb[w2i[q]]
        top = sims.argsort(descending=True)[:5]
        print(q, '->', [i2w[int(t)] for t in top])


### embeddings are tiny since the toy corpus is tiny. need to scale to text8 for real similarities.


### todo: subsampling, larger window, full text8.


### compared with pretrained glove: glove vectors capture much richer relations because text8 is big.
